# UNSW-NB15 — Task 3: Improvement, Ablation, CV, Significance Test
Group 5 · CSE475 · Track 1 (GNN)

Covers: extra ablation runs, 5-fold host-grouped CV on the baseline, and a McNemar
significance test between the final GNN and the best baseline — the two aggregated
to the same 47 test hosts so the comparison is actually paired and valid.

In [1]:
try:
    import torch_geometric
    print("torch_geometric already installed:", torch_geometric.__version__)
except ImportError:
    import torch
    TORCH_VER = torch.__version__.split('+')[0]
    CUDA_VER = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
    import os
    os.system(f"pip install -q torch_geometric")
    os.system(f"pip install -q pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html")
    import torch_geometric
    print("Installed torch_geometric:", torch_geometric.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 50.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 114.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 109.6 MB/s eta 0:00:00
Installed torch_geometric: 2.8.0.post1


In [2]:
import os, pickle, copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from statsmodels.stats.contingency_tables import mcnemar
import xgboost as xgb
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, GCNConv

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CFG = {
    'prep_dir': '/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset',
    'graph_dir': '/kaggle/input/datasets/shahiismyname/unsw-nb15-gnn-dataset',
    'baseline_dir': '/kaggle/input/datasets/shahiismyname/baseline-output',   # EDIT — Juju's task2_baseline_outputs, published as a dataset
    'gnn_dir': '/kaggle/input/datasets/shahiismyname/gnn-output',             # EDIT — Tithi's task2_gnn_outputs, published as a dataset

    'flow_id_col': 'flow_id', 'flow_host_cols': ['srcip', 'dstip'], 'flow_label_col': 'Label',
    'node_id_col': 'host', 'edge_feature_cols': ['transition_count', 'time_gap', 'proto_changed',
                                                   'port_changed', 'byte_diff', 'pkt_diff'],
    'node_feature_cols': ['avg_duration', 'total_bytes', 'num_connections',
                           'num_as_src', 'num_as_dst', 'num_unique_partners'],
    'heads': 2, 'hidden_dim': 8, 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 5e-4,
    'epochs': 200, 'patience': 20, 'val_frac': 0.2,

    'out_dir': '/kaggle/working/task3_outputs',
}
os.makedirs(CFG['out_dir'], exist_ok=True)

In [3]:
def load_pkl(folder, name):
    with open(os.path.join(folder, f'{name}.pkl'), 'rb') as f:
        return pickle.load(f)

train_processed = load_pkl(CFG['prep_dir'], 'train_processed')
test_processed = load_pkl(CFG['prep_dir'], 'test_processed')
side_train = load_pkl(CFG['prep_dir'], 'side_train')
side_test = load_pkl(CFG['prep_dir'], 'side_test')
fitted_transforms = load_pkl(CFG['prep_dir'], 'fitted_transforms')

nodes_train = load_pkl(CFG['graph_dir'], 'nodes_train')
nodes_test = load_pkl(CFG['graph_dir'], 'nodes_test')
edges_temporal_train = load_pkl(CFG['graph_dir'], 'edges_temporal_train')
edges_temporal_test = load_pkl(CFG['graph_dir'], 'edges_temporal_test')
edges_contact_train = load_pkl(CFG['graph_dir'], 'edges_contact_train')
edges_contact_test = load_pkl(CFG['graph_dir'], 'edges_contact_test')

le = load_pkl(CFG['baseline_dir'], 'label_encoder')
with open(os.path.join(CFG['baseline_dir'], 'XGBoost_model.pkl'), 'rb') as f:
    xgb_model = pickle.load(f)
baseline_preds = load_pkl(CFG['baseline_dir'], 'XGBoost_test_predictions')  # EDIT name if best model differs

gat_state = torch.load(os.path.join(CFG['gnn_dir'], 'gat_temporal_model.pt'), map_location=device)
gnn_preds = load_pkl(CFG['gnn_dir'], 'gat_temporal_test_predictions')

print("Loaded all files OK")

Loaded all files OK


In [4]:
def derive_host_labels(side_df, processed_df, flow_id_col, flow_host_cols, label_col):
    flow_labels = side_df[[flow_id_col] + flow_host_cols].merge(
        processed_df[[flow_id_col, label_col]], on=flow_id_col, how='inner')
    long = pd.concat([
        flow_labels[[flow_host_cols[0], label_col]].rename(columns={flow_host_cols[0]: 'host'}),
        flow_labels[[flow_host_cols[1], label_col]].rename(columns={flow_host_cols[1]: 'host'}),
    ], ignore_index=True)
    return long.groupby('host')[label_col].max().rename('host_label').reset_index()

host_labels_test = derive_host_labels(side_test, test_processed,
                                       CFG['flow_id_col'], CFG['flow_host_cols'], CFG['flow_label_col'])
print(host_labels_test.shape, "test hosts")

(47, 2) test hosts


In [5]:
# XGBoost predicts attack_cat (multiclass) — collapse to binary: anything not "Normal" = attack
normal_idx = list(le.classes_).index('Normal')
baseline_pred_binary = (baseline_preds['y_pred'] != normal_idx).astype(int)

flow_to_host = side_test[[CFG['flow_id_col']] + CFG['flow_host_cols']].merge(
    pd.DataFrame({CFG['flow_id_col']: baseline_preds['flow_id'], 'pred_binary': baseline_pred_binary}),
    on=CFG['flow_id_col'], how='inner')

long = pd.concat([
    flow_to_host[[CFG['flow_host_cols'][0], 'pred_binary']].rename(columns={CFG['flow_host_cols'][0]: 'host'}),
    flow_to_host[[CFG['flow_host_cols'][1], 'pred_binary']].rename(columns={CFG['flow_host_cols'][1]: 'host'}),
], ignore_index=True)

baseline_host_pred = long.groupby('host')['pred_binary'].max().rename('baseline_pred').reset_index()
print(baseline_host_pred.shape, "hosts with an aggregated baseline prediction")

(47, 2) hosts with an aggregated baseline prediction


In [6]:
compare_df = (host_labels_test
              .merge(baseline_host_pred, on='host', how='inner')
              .merge(gnn_preds[['host', 'y_pred']].rename(columns={'y_pred': 'gnn_pred'}), on='host', how='inner'))

assert len(compare_df) == len(host_labels_test), "host mismatch between baseline and GNN predictions — check flow_id/host alignment"

compare_df['baseline_correct'] = (compare_df['baseline_pred'] == compare_df['host_label']).astype(int)
compare_df['gnn_correct'] = (compare_df['gnn_pred'] == compare_df['host_label']).astype(int)

# McNemar 2x2 table: [[both correct, baseline correct & GNN wrong], [baseline wrong & GNN correct, both wrong]]
both_correct = ((compare_df['baseline_correct'] == 1) & (compare_df['gnn_correct'] == 1)).sum()
base_only = ((compare_df['baseline_correct'] == 1) & (compare_df['gnn_correct'] == 0)).sum()
gnn_only = ((compare_df['baseline_correct'] == 0) & (compare_df['gnn_correct'] == 1)).sum()
both_wrong = ((compare_df['baseline_correct'] == 0) & (compare_df['gnn_correct'] == 0)).sum()

table = [[both_correct, base_only], [gnn_only, both_wrong]]
print("McNemar contingency table:")
print(pd.DataFrame(table, index=['baseline correct', 'baseline wrong'],
                    columns=['GNN correct', 'GNN wrong']))

result = mcnemar(table, exact=True)  # exact binomial — appropriate given small n=47
print(f"\nMcNemar statistic: {result.statistic:.4f}")
print(f"p-value: {result.pvalue:.4f}")
print("Significant at alpha=0.05:" , "YES — models differ" if result.pvalue < 0.05 else "NO — no significant difference")
print(f"\n(n={len(compare_df)} paired hosts — small sample, note as a power limitation in the report)")

McNemar contingency table:
                  GNN correct  GNN wrong
baseline correct           38          9
baseline wrong              0          0

McNemar statistic: 0.0000
p-value: 0.0039
Significant at alpha=0.05: YES — models differ

(n=47 paired hosts — small sample, note as a power limitation in the report)


In [7]:
train_hosts_map = side_train[[CFG['flow_id_col']] + CFG['flow_host_cols']].copy()
train_hosts_map['host'] = train_hosts_map[CFG['flow_host_cols'][0]]

train_with_host = train_processed.merge(train_hosts_map[[CFG['flow_id_col'], 'host']], on=CFG['flow_id_col'])

drop_cols = [CFG['flow_id_col'], 'attack_cat', CFG['flow_label_col'], 'host']
X = train_with_host.drop(columns=[c for c in drop_cols if c in train_with_host.columns])
y = le.transform(train_with_host['attack_cat'])
groups = train_with_host['host']

# ct_ftp_cmd (and possibly others) can hold literal ' ' instead of a number in the raw data —
# same fix Juju applied in the baseline notebook, needed again here since this cell loads
# train_processed fresh
for col in X.columns:
    if X[col].dtype == object:
        X[col] = pd.to_numeric(X[col].astype(str).str.strip(), errors='coerce').fillna(0)
        print(f"Coerced {col} to numeric")

raw_weights = fitted_transforms['attack_cat_class_weights']
class_weight_dict = {le.transform([k])[0]: v**0.5 for k, v in raw_weights.items() if k in le.classes_}
sample_weight = np.array([class_weight_dict[label] for label in y])

gkf = GroupKFold(n_splits=5)
cv_scores = []
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    model = xgb.XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                               objective='multi:softprob', num_class=len(le.classes_),
                               tree_method='hist', n_jobs=-1, random_state=SEED)
    model.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=sample_weight[tr_idx])
    pred = model.predict(X.iloc[val_idx])
    cv_scores.append({
        'fold': fold,
        'f1_macro': f1_score(y[val_idx], pred, average='macro', zero_division=0),
        'accuracy': accuracy_score(y[val_idx], pred),
    })
    print(f"Fold {fold}: f1_macro={cv_scores[-1]['f1_macro']:.4f}  accuracy={cv_scores[-1]['accuracy']:.4f}")

cv_df = pd.DataFrame(cv_scores)
print(f"\nXGBoost 5-fold host-grouped CV: f1_macro = {cv_df['f1_macro'].mean():.4f} ± {cv_df['f1_macro'].std():.4f}")
print(f"accuracy = {cv_df['accuracy'].mean():.4f} ± {cv_df['accuracy'].std():.4f}")

Coerced ct_ftp_cmd to numeric
Fold 0: f1_macro=0.5830  accuracy=0.9708
Fold 1: f1_macro=0.5673  accuracy=0.9780
Fold 2: f1_macro=0.6168  accuracy=0.9768
Fold 3: f1_macro=0.6018  accuracy=0.9818
Fold 4: f1_macro=0.2000  accuracy=0.9999

XGBoost 5-fold host-grouped CV: f1_macro = 0.5138 ± 0.1764
accuracy = 0.9815 ± 0.0110


In [8]:
class GAT(nn.Module):
    def __init__(self, in_dim, edge_dim, hidden_dim, out_dim, heads=4, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.gat1 = GATv2Conv(in_dim, hidden_dim, heads=heads, dropout=dropout, edge_dim=edge_dim)
        self.gat2 = GATv2Conv(hidden_dim * heads, out_dim, heads=1, concat=False,
                               dropout=dropout, edge_dim=edge_dim)

    def forward(self, x, edge_index, edge_attr):
        x = F.elu(self.gat1(x, edge_index, edge_attr))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gat2(x, edge_index, edge_attr)
        return x


class GCNModel(nn.Module):
    """Ablation variant — GCN instead of GAT, ignores edge_attr (GCNConv doesn't take it)."""
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.gcn1 = GCNConv(in_dim, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, out_dim)

    def forward(self, x, edge_index, edge_attr=None):
        x = F.elu(self.gcn1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gcn2(x, edge_index)
        return x


def get_class_weights(y, device):
    classes, counts = torch.unique(y, return_counts=True)
    weights = torch.sqrt(len(y) / (len(classes) * counts.float()))
    full = torch.ones(int(classes.max()) + 1)
    for c, w in zip(classes, weights):
        full[c] = w
    return full.to(device)


def train_model(model_class, model_kwargs, train_data, cfg, verbose=False):
    model = model_class(**model_kwargs).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    data = train_data.to(device)
    class_weights = get_class_weights(data.y[data.train_mask], device)

    best_val_loss, best_state, patience_ctr = float('inf'), None, 0
    for epoch in range(cfg['epochs']):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, getattr(data, 'edge_attr', None))
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=class_weights)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index, getattr(data, 'edge_attr', None))
            val_loss = F.cross_entropy(out_eval[data.val_mask], data.y[data.val_mask], weight=class_weights).item()
        if val_loss < best_val_loss:
            best_val_loss, best_state, patience_ctr = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            patience_ctr += 1
        if patience_ctr >= cfg['patience']:
            break
    model.load_state_dict(best_state)
    return model


def eval_model(model, test_data):
    model.eval()
    data = test_data.to(device)
    with torch.no_grad():
        out = model(data.x, data.edge_index, getattr(data, 'edge_attr', None))
        probs = F.softmax(out, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
    y_true = data.y.cpu().numpy()
    return f1_score(y_true, preds, average='macro', zero_division=0), roc_auc_score(y_true, probs[:, 1])


def build_graph(nodes_df, edges_df, host_labels_df, node_feature_cols, edge_feature_cols, val_frac, seed=None):
    nodes_df = nodes_df.merge(host_labels_df, left_on='host', right_on='host', how='left')
    nodes_df['host_label'] = nodes_df['host_label'].fillna(0).astype(int)
    nodes_df = nodes_df.sort_values('node_index').reset_index(drop=True)

    x = torch.tensor(nodes_df[node_feature_cols].fillna(0).values, dtype=torch.float)
    y = torch.tensor(nodes_df['host_label'].values, dtype=torch.long)
    edge_index = torch.tensor(np.vstack([edges_df['source_host_idx'].values, edges_df['target_host_idx'].values]),
                               dtype=torch.long)
    if edge_feature_cols and all(c in edges_df.columns for c in edge_feature_cols):
        edge_attr = torch.tensor(edges_df[edge_feature_cols].fillna(0).values, dtype=torch.float)
    else:
        edge_attr = torch.ones((edge_index.shape[1], len(CFG['edge_feature_cols'])), dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    if seed is not None:
        g = torch.Generator().manual_seed(seed)
        n = data.num_nodes
        perm = torch.randperm(n, generator=g)
        n_val = max(1, int(n * val_frac))
        val_mask = torch.zeros(n, dtype=torch.bool); val_mask[perm[:n_val]] = True
        train_mask = torch.zeros(n, dtype=torch.bool); train_mask[perm[n_val:]] = True
        data.train_mask, data.val_mask = train_mask, val_mask
    return data


host_labels_train = derive_host_labels(side_train, train_processed,
                                        CFG['flow_id_col'], CFG['flow_host_cols'], CFG['flow_label_col'])

from sklearn.preprocessing import StandardScaler
node_scaler = StandardScaler()
nodes_train[CFG['node_feature_cols']] = node_scaler.fit_transform(nodes_train[CFG['node_feature_cols']])
nodes_test[CFG['node_feature_cols']] = node_scaler.transform(nodes_test[CFG['node_feature_cols']])
edge_scaler = StandardScaler()
edges_temporal_train[CFG['edge_feature_cols']] = edge_scaler.fit_transform(edges_temporal_train[CFG['edge_feature_cols']])
edges_temporal_test[CFG['edge_feature_cols']] = edge_scaler.transform(edges_temporal_test[CFG['edge_feature_cols']])

train_data = build_graph(nodes_train, edges_temporal_train, host_labels_train,
                          CFG['node_feature_cols'], CFG['edge_feature_cols'], CFG['val_frac'], seed=SEED)
test_data = build_graph(nodes_test, edges_temporal_test, host_labels_test,
                         CFG['node_feature_cols'], CFG['edge_feature_cols'], CFG['val_frac'])
print("Graph rebuilt for ablation runs.")

Graph rebuilt for ablation runs.


In [9]:
ablation_results = []

for dropout_val in [0.0, 0.3]:
    m = train_model(GAT, dict(in_dim=train_data.x.shape[1], edge_dim=train_data.edge_attr.shape[1],
                               hidden_dim=CFG['hidden_dim'], out_dim=2, heads=CFG['heads'], dropout=dropout_val),
                     train_data, CFG)
    f1, auc = eval_model(m, test_data)
    ablation_results.append({'ablation': f'GAT dropout={dropout_val}', 'f1_macro': f1, 'roc_auc': auc})

m_gcn = train_model(GCNModel, dict(in_dim=train_data.x.shape[1], hidden_dim=CFG['hidden_dim'],
                                    out_dim=2, dropout=CFG['dropout']), train_data, CFG)
f1, auc = eval_model(m_gcn, test_data)
ablation_results.append({'ablation': 'GCN (no edge features)', 'f1_macro': f1, 'roc_auc': auc})

ablation_df = pd.DataFrame(ablation_results)
print(ablation_df)
ablation_df.to_csv(os.path.join(CFG['out_dir'], 'extra_ablation_table.csv'), index=False)

                 ablation  f1_macro   roc_auc
0         GAT dropout=0.0  0.530000  0.632035
1         GAT dropout=0.3  0.530000  0.978355
2  GCN (no edge features)  0.507068  0.567100


In [10]:
compare_df.to_csv(os.path.join(CFG['out_dir'], 'mcnemar_host_comparison.csv'), index=False)
with open(os.path.join(CFG['out_dir'], 'mcnemar_result.pkl'), 'wb') as f:
    pickle.dump({'table': table, 'statistic': result.statistic, 'pvalue': result.pvalue}, f)
cv_df.to_csv(os.path.join(CFG['out_dir'], 'xgboost_5fold_cv.csv'), index=False)
print("Saved McNemar result, CV table, ablation table to", CFG['out_dir'])

Saved McNemar result, CV table, ablation table to /kaggle/working/task3_outputs
